# RoadSafe Delhi — Support Vector Machine (SVM) Model Development
## Assigned Member: Kaavish
### Task: Maximum-Margin Hyperplane Classification & Kernel Geometries

---

## 1. Project Introduction & Assigned Member
- **Project:** RoadSafe Delhi — Explainable Road Risk Mapping and Safety Intervention Prioritization Using Machine Learning
- **Assigned Member:** Kaavish
- **Model Assignment:** Support Vector Machine (`sklearn.svm.SVC`)
- **Role:** SVM model development, kernel selection (RBF vs Linear), domain literature research (3 papers), and boundary margin analysis.


## 2. Problem Statement & Model Objective
The objective is to train a Support Vector Classifier that establishes a maximum-margin separating hyperplane between high-risk road corridors and baseline road links.
- **Challenge:** Computational scaling of kernel matrices on large spatial networks and sensitivity to feature scaling.


## 3. Dataset Loading from Common Approved Dataset
Loading the shared spatial partitions from `data_final/`.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
DATA_FINAL_DIR = os.path.join(PROJECT_ROOT, "data_final")
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "models", "svm", "artifacts")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "models", "svm", "outputs")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

df_train = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "train.parquet"))
df_val = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "validation.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "test.parquet"))

print(f"Loaded Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")


## 4. Target Variable Selection
Selected Target: `is_crash_segment` (Binary Hazard Occurrence).


In [ ]:
target_col = "is_crash_segment"
print(f"Target: {target_col}")


## 5. Approved Feature Selection
Loading `feature_contract.json` and asserting zero target leakage.


In [ ]:
with open(os.path.join(DATA_FINAL_DIR, "feature_contract.json"), "r") as f:
    contract = json.load(f)

num_features = contract["approved_numerical_predictors"]
cat_features = contract["approved_categorical_predictors"]
approved_predictors = num_features + cat_features

forbidden = set(contract["quarantined_leakage_columns"]).union(set(contract["target_definitions"].keys()))
assert len(set(approved_predictors).intersection(forbidden)) == 0, "FATAL: Leakage detected!"
print("Approved Predictors:", approved_predictors)


## 6. Data Preprocessing (Mandatory Scaling for SVM)
SVM is strictly distance-dependent. Standardizing numerical features and one-hot encoding categorical features on training data.


In [ ]:
scaler = StandardScaler()
scaler.fit(df_train[num_features])

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df_train[cat_features])

X_train = np.hstack([scaler.transform(df_train[num_features]), encoder.transform(df_train[cat_features])])
X_val = np.hstack([scaler.transform(df_val[num_features]), encoder.transform(df_val[cat_features])])
X_test = np.hstack([scaler.transform(df_test[num_features]), encoder.transform(df_test[cat_features])])

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values
print(f"X_train scaled matrix: {X_train.shape}")


## 7. Train-Validation-Test Data Verification (Common Spatial Split)
Verifying disjoint spatial block boundaries.


In [ ]:
assert set(df_train['spatial_block']).isdisjoint(set(df_val['spatial_block']))
assert set(df_train['spatial_block']).isdisjoint(set(df_test['spatial_block']))
print("Spatial Separation Verified.")


## 8. Model Architecture & Configuration
Configuring `SVC` with RBF kernel and `probability=True` for probability calibration.


In [ ]:
svm_model = SVC(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)
print("Configured SVM Model:", svm_model)


## 9. Model Training
Training SVM on `X_train`, `y_train`.


In [ ]:
svm_model.fit(X_train, y_train)
print("SVM Training Completed.")


## 10. Validation & Hyperparameter Tuning
Tuning decision threshold on validation set to maximize F1-score.


In [ ]:
from sklearn.metrics import precision_recall_curve

y_val_prob = svm_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = np.zeros_like(thresholds)
for i, t in enumerate(thresholds):
    f1_scores[i] = f1_score(y_val, (y_val_prob >= t).astype(int), zero_division=0)

best_idx = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx])
print(f"Optimal Validation Threshold: {optimal_threshold:.4f} (Validation F1 = {f1_scores[best_idx]:.4f})")


## 11. Final Testing on the Untouched Test Dataset
Evaluating SVM on `X_test` with the frozen validation threshold.


In [ ]:
y_test_prob = svm_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

print(f"Test Accuracy:  {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print(f"Test Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test Recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test F1-Score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"Test PR-AUC:    {average_precision_score(y_test, y_test_prob):.4f}")


## 12. Evaluation Metrics & Visualizations
Plotting Confusion Matrix and ROC Curve.


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=ax)
ax.set_title(f"SVM Test Confusion Matrix (Threshold={optimal_threshold:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "confusion_matrix.png"), dpi=200)
plt.show()


## 13. Results Interpretation
Analysis of support vectors and margin separation performance across spatial corridors.


In [ ]:
print(f"Total Support Vectors: {svm_model.n_support_.sum()} ({svm_model.n_support_})")


## 14. Model and Artifact Saving
Saving SVM model, scaler, encoder, threshold, and metrics.


In [ ]:
import joblib

joblib.dump(svm_model, os.path.join(ARTIFACTS_DIR, "svm_model.joblib"))
joblib.dump(scaler, os.path.join(ARTIFACTS_DIR, "scaler.joblib"))
joblib.dump(encoder, os.path.join(ARTIFACTS_DIR, "encoder.joblib"))
with open(os.path.join(ARTIFACTS_DIR, "threshold.json"), "w") as f:
    json.dump({"optimal_threshold": optimal_threshold}, f, indent=2)

print(f"Saved SVM artifacts to {ARTIFACTS_DIR}")


## 15. Final Summary & Observations
Summary of SVM classification boundaries and literature review integration.
